User Input
      │
      ▼
Language Detection
      │
      ▼
FAISS Retriever
      │
      ▼
Retrieved Context
      │
      ▼
Prompt Building
      │
      ▼
Mistral
      │
      ▼
Travel Plan / Recommendations

User Input
      │
      ▼
Build Query
      │
      ▼
Retriever (FAISS)
      │
      ▼
Retrieved Documents
      │
      ▼
Build Context
      │
      ▼
Language Detection
      │
      ▼
Prompt Selection
      │
      ▼
Final Prompt
      │
      ▼
Mistral
      │
      ▼
Trip Plan
      │
      ▼
Save Result

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langdetect import detect

C:\Users\Administrator\AppData\Local\Temp\ipykernel_6480\2330158602.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding Model Loaded Successfully!")

Embedding Model Loaded Successfully!


In [3]:
vectorstore = FAISS.load_local(
    "egypt_faiss",
    embedding_model,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

print("FAISS Loaded Successfully!")

FAISS Loaded Successfully!


In [4]:
llm = ChatOllama(
    model="mistral",
    temperature=0.3,
    num_predict=300
)

print("Mistral Loaded Successfully!")

Mistral Loaded Successfully!


In [5]:
city = "Luxor"
days = 3
budget = "Medium"
people = 2
trip_type = "Friends"
interests = "Temples, Museums, History"

In [6]:
query = f"""
City: {city}

Interests:
{interests}
"""

docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content[:500]
    for doc in docs
)

print(f"Retrieved {len(docs)} Documents")

Retrieved 3 Documents


In [7]:
language = detect(interests)

print(language)

en


In [8]:
planner_prompt = """
You are an Egypt Tourism Assistant.

Use ONLY the provided context.

Rules:
- Use only the context.
- Do not invent information.
- Do not recommend places not found in the context.
- If information is missing, write:
  "Not available in the knowledge base."
- If there are not enough attractions for all requested days, write:
  "No additional attractions are available in the knowledge base."
- Answer in the same language as the user.

Context:
{context}

City: {city}
Days: {days}
Interests: {interests}

Create a day-by-day itinerary.

For each day include:
- Attraction
- Why Visit
- Visit Duration (if available)
- Notes
"""

In [9]:
planner_prompt_ar = """
أنت مساعد ذكي للسياحة في مصر.

استخدم المعلومات الموجودة في السياق فقط.

القواعد:
- لا تستخدم أي معلومات خارج السياق.
- لا تخترع أي معلومات.
- لا تقترح أماكن غير موجودة في السياق.
- إذا كانت المعلومة غير موجودة اكتب:
  "غير متوفر في قاعدة المعرفة."
- إذا لم تكفِ الأماكن لجميع الأيام اكتب:
  "لا توجد أماكن إضافية متوفرة في قاعدة المعرفة."
- أجب بالعربية فقط.

السياق:
{context}

المدينة: {city}
عدد الأيام: {days}
الاهتمامات: {interests}

أنشئ برنامجًا سياحيًا يوميًا.

لكل يوم اذكر:
- المكان
- سبب الزيارة
- مدة الزيارة (إن وجدت)
- ملاحظات
"""

In [10]:
city = "Luxor"
days = 3
interests = "Temples and History"


In [11]:
query = f"""
City: {city}

Interests:
{interests}
"""

print(query)


City: Luxor

Interests:
Temples and History



In [12]:
docs = retriever.invoke(query)

print(f"Retrieved {len(docs)} documents")

Retrieved 3 documents


In [13]:
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

print(context)

Historical Description:
Luxor Temple sits directly in the heart of modern Luxor city, on the east bank of the Nile. Unlike most Egyptian temples dedicated to a specific deity for eternal worship, Luxor Temple was largely dedicated to the rejuvenation of kingship, closely tied to the annual Opet Festival, during which statues of the Theban triad (Amun, Mut, and Khonsu) were ceremonially carried from Karnak to Luxor Temple to reaffirm the pharaoh's divine right to rule. The temple has been in continuous use in some form for over 3,400 years, later incorporating a Roman-era chapel, a mosque (Abu al-Haggag Mosque, still active today), and even earlier a Coptic church, layered atop the ancient structure.

Question:
Is Luxor Temple located inside the city?

Answer:
Yes, uniquely among major Egyptian monuments, Luxor Temple sits directly within the modern city center of Luxor, surrounded by streets, shops, and the Nile corniche.

Question:
Is Luxor Temple connected to Karnak Temple?

Answer:


In [14]:
language = detect(interests)

print(language)

en


In [15]:
if language == "ar":
    prompt = planner_prompt_ar
else:
    prompt = planner_prompt

In [16]:
final_prompt = prompt.format(
    context=context,
    city=city,
    days=days,
    interests=interests
)

In [17]:
print("Generating Trip Plan...")

response = llm.invoke(final_prompt)

trip_plan = response.content

Generating Trip Plan...


In [18]:
print(trip_plan)

 Day 1: Luxor Temple
Attractions: Luxor Temple
Why Visit: Explore the ancient temple dedicated to the rejuvenation of kingship, closely tied to the annual Opet Festival. Witness the layers of history from Ancient Egyptian, Roman, and Coptic periods.
Visit Duration: Approximately 2-3 hours
Notes: The temple is located in the heart of modern Luxor city, making it easily accessible for visitors.

Day 2: Karnak Temple
Attractions: Karnak Temple Complex
Why Visit: Marvel at the largest temple complex ever built by man, housing the Great Hypostyle Hall with its massive columns and intricate carvings.
Visit Duration: Approximately 3-4 hours
Notes: Connect to Luxor Temple via the Avenue of Sphinxes, a processional route used during the Opet Festival.

Day 3: Valley of the Kings
Attractions: Valley of the Kings
Why Visit: Discover the tombs of Pharaohs including Tutankhamun and Ramses II, offering insights into ancient Egyptian burial rituals and artistry.
Visit Duration: Approximately 2-3 hour

In [19]:
trip_result = {
    "city": city,
    "days": days,
    "interests": interests,
    "trip_plan": trip_plan,
    "sources": [
        doc.metadata
        for doc in docs
    ]
}

trip_result

{'city': 'Luxor',
 'days': 3,
 'interests': 'Temples and History',
 'trip_plan': ' Day 1: Luxor Temple\nAttractions: Luxor Temple\nWhy Visit: Explore the ancient temple dedicated to the rejuvenation of kingship, closely tied to the annual Opet Festival. Witness the layers of history from Ancient Egyptian, Roman, and Coptic periods.\nVisit Duration: Approximately 2-3 hours\nNotes: The temple is located in the heart of modern Luxor city, making it easily accessible for visitors.\n\nDay 2: Karnak Temple\nAttractions: Karnak Temple Complex\nWhy Visit: Marvel at the largest temple complex ever built by man, housing the Great Hypostyle Hall with its massive columns and intricate carvings.\nVisit Duration: Approximately 3-4 hours\nNotes: Connect to Luxor Temple via the Avenue of Sphinxes, a processional route used during the Opet Festival.\n\nDay 3: Valley of the Kings\nAttractions: Valley of the Kings\nWhy Visit: Discover the tombs of Pharaohs including Tutankhamun and Ramses II, offering in

In [20]:
%whos

Variable                Type                     Data/Info
----------------------------------------------------------
ChatOllama              ModelMetaclass           <class 'langchain_ollama.chat_models.ChatOllama'>
FAISS                   ABCMeta                  <class 'langchain_communi<...>ectorstores.faiss.FAISS'>
HuggingFaceEmbeddings   ModelMetaclass           <class 'langchain_hugging<...>e.HuggingFaceEmbeddings'>
budget                  str                      Medium
city                    str                      Luxor
context                 str                      Historical Description:\n<...>tored for visitors today.
days                    int                      3
detect                  function                 <function detect at 0x000001B1A63A5440>
docs                    list                     n=3
embedding_model         HuggingFaceEmbeddings    model_name='sentence-tran<...>False show_progress=False
final_prompt            str                      \nYou are 

In [21]:
def plan_trip(city, days, interests):

    query = f"""
    City: {city}

    Interests:
    {interests}
    """

    docs = retriever.invoke(query)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    language = detect(city + " " + interests)

    if language == "ar":
        prompt = planner_prompt_ar
    else:
        prompt = planner_prompt

    final_prompt = prompt.format(
        context=context,
        city=city,
        days=days,
        interests=interests
    )

    response = llm.invoke(final_prompt)

    return response.content

In [22]:
print(
    plan_trip(
        city="Luxor",
        days=2,
        interests="History and Mediterranean Culture"
    )
)

 Day 1:
- Attraction: Luxor Temple
  - Why Visit: This ancient temple is unique as it sits within the modern city center of Luxor. It was dedicated to the rejuvenation of kingship and has been in continuous use for over 3,400 years, incorporating a Roman-era chapel, a mosque, and earlier a Coptic church.
  - Visit Duration: Approximately 2 hours
  - Notes: The temple is located in the city center, surrounded by streets, shops, and the Nile corniche.

Day 2:
- Attraction: Karnak Temple Complex
  - Why Visit: Karnak Temple Complex is one of the largest religious buildings ever constructed, dating back to the Middle Kingdom. It was dedicated to the Theban triad (Amun, Mut, and Khonsu), who were also worshipped at Luxor Temple.
  - Visit Duration: Approximately 3 hours
  - Notes: Karnak Temple Complex is located approximately 2.5 kilometers north of Luxor Temple. It's a must-see for anyone interested in ancient Egyptian history and culture.


In [24]:
print(
    plan_trip(
        city="الأقصر",
        days=3,
        interests="المعابد والتاريخ"
    )
)

 برنامج سياحي يومي:

**يوم أول:**
- المكان: معبد الكرنك (Luxor, Upper Egypt, East Bank of the Nile)
- سبب الزيارة: زيارة منطقة تحتوي على أكثر من 1300 يامناً فيها أكبر كليسة في العالم.
- مدة الزيارة: 4 ساعات (حد أدنى)
- ملاحظات: يوجد خدمات للطاقة والبيتزا في المكان.

**يوم ثاني:**
- المكان: وادي الملوك (Luxor, Upper Egypt, West Bank of the Nile opposite Luxor)
- سبب الزيارة: زيارة مقعد الفرائض المصرية التي 
